In [1]:
!pip install torchtext spacy datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.3 MB/s eta 0:00:00


In [2]:
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 113.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import time
import spacy
from collections import Counter
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device is {device}")

device is cuda


In [4]:
!pip install datasets==2.14.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.2/492.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 11.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [5]:
import urllib.request
import os

In [6]:
os.makedirs("data", exist_ok=True)
base_url = "https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/"

In [7]:
files = {"train.txt":"eng.train", "valid.txt":"eng.testa", "test.txt":"eng.testb"}

In [8]:
type(files.items())

dict_items

In [9]:
for local_name, remote_name in files.items():
  path = f"data/{local_name}"
  if True:
    print(f"downloading {remote_name}")
    urllib.request.urlretrieve(f"{base_url}{remote_name}", path)


downloading eng.train
downloading eng.testa
downloading eng.testb


In [10]:
with open("data/train.txt", 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        line = line.strip()
        print(line)
        if i>10:
          break


-DOCSTART- -X- O O

EU NNP I-NP I-ORG
rejects VBZ I-VP O
German JJ I-NP I-MISC
call NN I-NP O
to TO I-VP O
boycott VB I-VP O
British JJ I-NP I-MISC
lamb NN I-NP O
. . O O



In [11]:
def load_conll(file_path):
  sentences = []
  labels = []
  current_tokens = []
  current_labels = []
  with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
      line = line.strip()
      if line == "" or line.startswith("-DOCSTART-"):
        if current_tokens:
          sentences.append(current_tokens)
          labels.append(current_labels)
          current_tokens = []
          current_labels = []
        continue
      parts = line.split()
      current_tokens.append(parts[0])
      current_labels.append(parts[3])

    if current_tokens:
      sentences.append(current_tokens)
      labels.append(current_labels)

  return sentences, labels




In [12]:
train_sentences, train_labels = load_conll("data/train.txt")
val_sentences, val_labels = load_conll("data/valid.txt")
test_sentences, test_labels = load_conll("data/test.txt")


In [13]:
train_sentences[:1]

[['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']]

In [14]:
train_labels[:1]

[['I-ORG', 'O', 'I-MISC', 'O', 'O', 'O', 'I-MISC', 'O', 'O']]

In [15]:
print(f"Train: {len(train_sentences)} sentences")
print(f"Val:   {len(val_sentences)} sentences")
print(f"Test:  {len(test_sentences)} sentences")

print(f"\nExample:")
print(f"Tokens: {train_sentences[0]}")
print(f"Labels: {train_labels[0]}")

Train: 14041 sentences
Val:   3250 sentences
Test:  3453 sentences

Example:
Tokens: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Labels: ['I-ORG', 'O', 'I-MISC', 'O', 'O', 'O', 'I-MISC', 'O', 'O']


In [16]:
def convert_to_iob2(labels):
    new_labels = []
    for i, label in enumerate(labels):
        if label.startswith("I-"):
            # If it's the first token OR previous token has a different entity type
            # then it should be B- not I-
            if i == 0 or labels[i-1] == "O" or labels[i-1].split("-")[1] != label.split("-")[1]:
                new_labels.append("B-" + label.split("-")[1])
            else:
                new_labels.append(label)
        else:
            new_labels.append(label)
    return new_labels

In [17]:
train_labels = [convert_to_iob2(label) for label in train_labels]
val_labels = [convert_to_iob2(label) for label in val_labels]
test_labels = [convert_to_iob2(label) for label in test_labels]

In [18]:
label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
label2idx = {label:idx for idx,label in enumerate(label_list)}
idx2label = {idx:label for label,idx in label2idx.items()}

In [19]:
NUM_LABELS = len(label_list)

In [20]:
print(f"Number of labels: {NUM_LABELS}")
print(f"\nLabel mappings:")
for label, idx in label2idx.items():
    print(f"  {label:8s} → {idx}")

# Verify conversion
print(f"\nBefore conversion example: ['I-ORG', 'O', 'I-MISC', ...]")
print(f"After conversion example:  {train_labels[0]}")

Number of labels: 9

Label mappings:
  O        → 0
  B-PER    → 1
  I-PER    → 2
  B-ORG    → 3
  I-ORG    → 4
  B-LOC    → 5
  I-LOC    → 6
  B-MISC   → 7
  I-MISC   → 8

Before conversion example: ['I-ORG', 'O', 'I-MISC', ...]
After conversion example:  ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']


In [21]:
def build_vocab(senetnces, min_frequency=2):
  counter = Counter()
  for sent in senetnces:
    counter.update(sent)
  token2idx = {'<pad>':0, '<unk>':1}
  for word, freq in counter.items():
    if freq>=min_frequency and word not in token2idx:
      token2idx.update({word:len(token2idx)})

  idx2token = {idx:word for word,idx in token2idx.items()}
  return token2idx, idx2token

In [22]:
PAD_IDX = 0
UNK_IDX = 1

In [23]:
token2idx, idx2token = build_vocab(train_sentences, min_frequency=1)

In [24]:
print(f"Vocab size: {len(token2idx)}")
print(f"\nSample mappings:")
for word in ['<pad>', '<unk>', 'EU', 'German', 'the', 'Obama']:
    print(f"  '{word}' → {token2idx.get(word, 'NOT FOUND')}")

Vocab size: 23625

Sample mappings:
  '<pad>' → 0
  '<unk>' → 1
  'EU' → 2
  'German' → 4
  'the' → 41
  'Obama' → NOT FOUND


In [25]:
class NERDataset(Dataset):
  def __init__(self, sentences, labels, token2idx, label2idx):
    self.sentences = sentences
    self.labels = labels
    self.token2idx = token2idx
    self.label2idx = label2idx

  def __len__(self):
    return len(self.sentences)

  def __getitem__(self, idx):
    sentence = self.sentences[idx]
    label = self.labels[idx]
    token_indices = [token2idx.get(sent, UNK_IDX) for sent in sentence]
    label_indices = [label2idx.get(lab) for lab in label]
    return torch.tensor(token_indices, dtype=torch.long), torch.tensor(label_indices, dtype=torch.long)




In [26]:
train_dataset = NERDataset(train_sentences, train_labels, token2idx, label2idx)
val_dataset = NERDataset(val_sentences, val_labels, token2idx, label2idx)

In [27]:
tokens, labels = train_dataset[0]
print(f"Token indices: {tokens}")
print(f"Label indices: {labels}")
print(f"Token length:  {len(tokens)}")
print(f"Label length:  {len(labels)}")

Token indices: tensor([ 2,  3,  4,  5,  6,  7,  8,  9, 10])
Label indices: tensor([3, 0, 7, 0, 0, 0, 7, 0, 0])
Token length:  9
Label length:  9


In [28]:
t1 = [torch.tensor([2, 3, 4]), torch.tensor([5, 6, 7])]
torch.stack(t1)

tensor([[2, 3, 4],
        [5, 6, 7]])

In [29]:
LABEL_PAD_IDX = -100

In [30]:
l1 = [torch.tensor([1,2,3]), torch.tensor([2,3,5,8])]
l2 = [len(tok) for tok in l1]
t1 = (len(tok) for tok in l1)

In [31]:
l2

[3, 4]

In [32]:
t1

<generator object <genexpr> at 0x7df1bdcb6b50>

In [33]:
list(zip([1,2,3],[4,5,6]))

[(1, 4), (2, 5), (3, 6)]

In [34]:
t1 = torch.tensor([1,2,3])
t2 = torch.tensor([0,0,0])
torch.cat([t1,t2])

tensor([1, 2, 3, 0, 0, 0])

In [35]:
def collate_fn(batch):
  tokens_batch, labels_batch = zip(*batch)
  max_len = max(len(tok) for tok in tokens_batch)
  tokens_padded = []
  labels_padded = []
  for tokens, labels in zip(tokens_batch, labels_batch):
    token_padding = torch.full((max_len-len(tokens),), PAD_IDX, dtype=torch.long)
    tokens_padded.append(torch.cat([tokens, token_padding]))
    labels_padding = torch.full((max_len-len(labels),), LABEL_PAD_IDX, dtype=torch.long)
    labels_padded.append(torch.cat([labels, labels_padding]))

  tokens_batch = torch.stack(tokens_padded)
  labels_batch = torch.stack(labels_padded)
  return tokens_batch, labels_batch

In [36]:
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [37]:
tokens, labels = next(iter(train_loader))

In [38]:
print(f"Tokens batch shape: {tokens.shape}")
print(f"Labels batch shape: {labels.shape}")

Tokens batch shape: torch.Size([64, 43])
Labels batch shape: torch.Size([64, 43])


In [39]:
class TokenEmbedding(nn.Module):
  def __init__(self, vocab_size, d_model):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, d_model)
    self.d_model = d_model

  def forward(self, x):
    return self.embedding(x) * math.sqrt(self.d_model)

In [40]:
test_embed = TokenEmbedding(vocab_size=23625, d_model=256)
dummy = torch.tensor([[1, 21, 91, 2]])
output = test_embed(dummy)
print(f"Input shape:  {dummy.shape}")    # (1, 4)
print(f"Output shape: {output.shape}")   # (1, 4, 256)

Input shape:  torch.Size([1, 4])
Output shape: torch.Size([1, 4, 256])


In [41]:
pe = torch.zeros(3, 10)

In [42]:
type(pe)

torch.Tensor

In [43]:
pe

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [44]:
pe.shape

torch.Size([3, 10])

In [45]:
pe.squeeze()

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [46]:
pe.unsqueeze(0)

tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]])

In [47]:
pe.shape

torch.Size([3, 10])

In [48]:
pe.unsqueeze(1).shape

torch.Size([3, 1, 10])

In [49]:
class  PostionalEncoding(nn.Module):
  def __init__(self, d_model, max_len=5000, dropout=0.1):
    super().__init__()
    self.dropout = nn.Dropout(dropout)
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0)/d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    pe = pe.unsqueeze(0)
    self.register_buffer('pe', pe)

  def forward(self, x):
    x = x + self.pe[:, :x.size(1), :]
    return self.dropout(x)


In [50]:
test_pe = PostionalEncoding(d_model=256)
dummy = torch.zeros(2, 10, 256)
output = test_pe(dummy)
print(f"Input shape:  {dummy.shape}")    # (2, 10, 256)
print(f"Output shape: {output.shape}")   # (2, 10, 256)

Input shape:  torch.Size([2, 10, 256])
Output shape: torch.Size([2, 10, 256])


In [51]:
import torch
t1 = torch.randn(2,3,6)

In [52]:
t1

tensor([[[ 1.1854, -1.1525,  0.8919, -0.6184, -0.3641,  0.6819],
         [-0.1098, -0.7293, -0.0194,  1.2087,  1.7931,  2.2961],
         [-0.5284, -0.3940, -1.3249,  0.3289,  1.8958,  0.0391]],

        [[-2.0686,  1.6915,  0.2610, -0.5258, -0.0637, -0.4166],
         [-0.0413, -1.0875, -0.0151,  0.6226,  0.7929, -0.2946],
         [ 0.5396, -0.3592, -1.4575,  0.2611, -2.4522,  0.1645]]])

In [53]:
t1.shape

torch.Size([2, 3, 6])

In [54]:
t2 = t1.view(2, -1, 2, 3)

In [55]:
t2.shape

torch.Size([2, 3, 2, 3])

In [56]:
 t2

tensor([[[[ 1.1854, -1.1525,  0.8919],
          [-0.6184, -0.3641,  0.6819]],

         [[-0.1098, -0.7293, -0.0194],
          [ 1.2087,  1.7931,  2.2961]],

         [[-0.5284, -0.3940, -1.3249],
          [ 0.3289,  1.8958,  0.0391]]],


        [[[-2.0686,  1.6915,  0.2610],
          [-0.5258, -0.0637, -0.4166]],

         [[-0.0413, -1.0875, -0.0151],
          [ 0.6226,  0.7929, -0.2946]],

         [[ 0.5396, -0.3592, -1.4575],
          [ 0.2611, -2.4522,  0.1645]]]])

In [57]:
t3 = t2.transpose(1,2)

In [58]:
t3.shape

torch.Size([2, 2, 3, 3])

In [59]:
t4 = t3.masked_fill(torch.tensor([False, True, False]), float('-inf'))

In [60]:
t4

tensor([[[[ 1.1854,    -inf,  0.8919],
          [-0.1098,    -inf, -0.0194],
          [-0.5284,    -inf, -1.3249]],

         [[-0.6184,    -inf,  0.6819],
          [ 1.2087,    -inf,  2.2961],
          [ 0.3289,    -inf,  0.0391]]],


        [[[-2.0686,    -inf,  0.2610],
          [-0.0413,    -inf, -0.0151],
          [ 0.5396,    -inf, -1.4575]],

         [[-0.5258,    -inf, -0.4166],
          [ 0.6226,    -inf, -0.2946],
          [ 0.2611,    -inf,  0.1645]]]])

In [61]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, n_heads):
    super().__init__()
    assert d_model % n_heads == 0
    self.d_model = d_model
    self.n_heads = n_heads
    self.d_k = d_model // n_heads
    self.W_q = nn.Linear(d_model, d_model)
    self.W_k = nn.Linear(d_model, d_model)
    self.W_v = nn.Linear(d_model, d_model)
    self.W_o = nn.Linear(d_model, d_model)

  def scaled_dot_product_attention(self, Q, K, V, mask=None):
    scores = torch.matmul(Q, K.transpose(-2, -1))/math.sqrt(self.d_k)
    if mask is not None:
      scores = scores.masked_fill(mask==0, float('-inf'))
    attn_weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output

  def forward(self, query, key, value, mask=None):
    batch_size = query.size(0)
    Q = self.W_q(query)
    K = self.W_k(key)
    V = self.W_v(value)
    Q = Q.view(batch_size, -1, self.n_heads, self.d_k).transpose(1,2)
    K = K.view(batch_size, -1, self.n_heads, self.d_k).transpose(1,2)
    V = V.view(batch_size, -1, self.n_heads, self.d_k).transpose(1,2)
    attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
    attn_output = attn_output.transpose(1,2).contiguous().view(batch_size, -1, self.d_model)
    output = self.W_o(attn_output)
    return output


In [62]:
test_mha = MultiHeadAttention(d_model=256, n_heads=8)
dummy = torch.randn(2, 10, 256)
output = test_mha(dummy, dummy, dummy)
print(f"Input shape:  {dummy.shape}")    # (2, 10, 256)
print(f"Output shape: {output.shape}")   # (2, 10, 256)

Input shape:  torch.Size([2, 10, 256])
Output shape: torch.Size([2, 10, 256])


In [63]:
class FeedForward(nn.Module):
  def __init__(self, d_model, d_ff, dropout=0.1):
    super().__init__()
    self.linear1 = nn.Linear(d_model, d_ff)
    self.linear2 = nn.Linear(d_ff, d_model)
    self.relu = nn.ReLU()
    self.dropout = nn.Dropout(dropout)


  def forward(self, x):
    x = self.linear1(x)
    x = self.relu(x)
    x = self.dropout(x)
    x = self.linear2(x)
    return x

In [64]:
test_ff = FeedForward(d_model=256, d_ff=1024)
dummy = torch.randn(2, 10, 256)
output = test_ff(dummy)
print(f"Input shape:  {dummy.shape}")    # (2, 10, 256)
print(f"Output shape: {output.shape}")   # (2, 10, 256)

Input shape:  torch.Size([2, 10, 256])
Output shape: torch.Size([2, 10, 256])


In [65]:
class EncoderLayer(nn.Module):
  def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
    super().__init__()
    self.self_attention = MultiHeadAttention(d_model, n_heads)
    self.feed_forward = FeedForward(d_model, d_ff, dropout)
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.dropout1 = nn.Dropout(dropout)
    self.dropout2 = nn.Dropout(dropout)

  def forward(self, x, mask):
    attn_output = self.self_attention(x, x, x, mask)
    attn_output = self.dropout1(attn_output)
    x = self.norm1(x+attn_output)
    ff_output = self.feed_forward(x)
    ff_output = self.dropout2(ff_output)
    x = self.norm2(x + ff_output)
    return x



In [66]:
class NERTransformer(nn.Module):
  def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, num_labels, dropout=0.1):
    super().__init__()
    self.token_embedding = TokenEmbedding(vocab_size, d_model)
    self.positional_encoding = PostionalEncoding(d_model, dropout=0.1)
    self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
    self.classifier = nn.Linear(d_model, num_labels)

  def forward(self, src, mask):
    x = self.token_embedding(src)
    x = self.positional_encoding(x)
    for layer in self.layers:
      x = layer(x, mask)

    output = self.classifier(x)
    return output

In [67]:
dummy = torch.randint(0, 1000, (2, 10))
test_model = NERTransformer(vocab_size=len(token2idx),
    d_model=256,
    n_heads=8,
    d_ff=1024,
    n_layers=3,
    num_labels=NUM_LABELS)
output = test_model(dummy, mask=None)
print(f"Input shape:  {dummy.shape}")    # (2, 10)
print(f"Output shape: {output.shape}")   # (2, 10, 9)

Input shape:  torch.Size([2, 10])
Output shape: torch.Size([2, 10, 9])


In [68]:
total_params = sum(p.numel() for p in test_model.parameters())

In [69]:
total_params

8419593

In [70]:
print(f"Parameters:   {total_params:,}")

Parameters:   8,419,593


In [71]:
def create_mask(src, pad_idx=PAD_IDX):
  mask = (src!=pad_idx).unsqueeze(1).unsqueeze(2)
  return mask

In [72]:
D_MODEL = 256
N_HEADS = 8
D_FF = 1024
N_LAYERS = 3
DROPOUT = 0.1
LEARNING_RATE = 3e-4
EPOCHS = 20

In [73]:
model = NERTransformer(vocab_size=len(token2idx), d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF, n_layers=N_LAYERS, num_labels=NUM_LABELS, dropout=DROPOUT).to(device)
loss_fn = nn.CrossEntropyLoss(ignore_index=LABEL_PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.98, 0.98), eps=1e-9)

In [74]:
total_params = sum(param.numel() for param in model.parameters())
total_params

8419593

In [75]:
print(f"Model parameters: {total_params:,}")
print(f"Device: {device}")

Model parameters: 8,419,593
Device: cuda


In [76]:
t11 = torch.randn(2,5,9)

In [77]:
t11.shape

torch.Size([2, 5, 9])

In [78]:
t12 = t11.reshape(-1, t11.size(-1))

In [79]:
t12.shape

torch.Size([10, 9])

In [80]:
def train_one_epoch(model, dataloader, optimizer, loss_fn, device):
  model.train()
  total_loss = 0
  total_tokens = 0
  for tokens, labels in dataloader:
    tokens = tokens.to(device)
    labels = labels.to(device)
    mask = create_mask(tokens).to(device)
    output = model(tokens, mask)

    output = output.reshape(-1, output.size(-1))
    labels = labels.reshape(-1)
    loss = loss_fn(output, labels)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    non_pad = (labels!=LABEL_PAD_IDX).sum().item()
    total_loss += loss.item()*non_pad
    total_tokens = total_tokens + non_pad
  return total_loss / total_tokens

In [81]:
def evaluate(model, dataloader, loss_fn, device):
  model.eval()
  total_loss = 0
  total_tokens = 0
  with torch.no_grad():
    for tokens, labels in dataloader:
      tokens = tokens.to(device)
      labels = labels.to(device)
      mask = create_mask(tokens).to(device)
      output = model(tokens, mask)
      output = output.reshape(-1, output.size(-1))
      labels = labels.reshape(-1)
      loss = loss_fn(output, labels)
      non_pad = (labels!=LABEL_PAD_IDX).sum().item()
      total_loss+=loss.item()*non_pad
      total_tokens+=non_pad
  return total_loss/total_tokens

In [82]:
def compute_accuracy(model, dataloader, device):
  model.eval()
  correct = 0
  total = 0
  with torch.no_grad():
    for tokens, labels in dataloader:
      tokens = tokens.to(device)
      labels = labels.to(device)
      mask = create_mask(tokens).to(device)
      output = model(tokens, mask)
      predictions = output.argmax(dim=-1)
      non_pad_mask = (labels!=LABEL_PAD_IDX)
      correct += (predictions[non_pad_mask]==labels[non_pad_mask]).sum().item()
      total += non_pad_mask.sum().item()

  return correct/total



In [83]:
for epoch in range(EPOCHS):
  start_time = time.time()
  train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
  val_loss = evaluate(model, val_loader, loss_fn, device)
  val_acc = compute_accuracy(model, val_loader, device)
  elapsed = time.time() - start_time
  print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Time: {elapsed:.1f}s")


Epoch 01/20 | Train Loss: 0.5981 | Val Loss: 0.4636 | Val Acc: 0.8734 | Time: 6.8s
Epoch 02/20 | Train Loss: 0.3699 | Val Loss: 0.3600 | Val Acc: 0.8959 | Time: 5.8s
Epoch 03/20 | Train Loss: 0.2725 | Val Loss: 0.3089 | Val Acc: 0.9085 | Time: 5.7s
Epoch 04/20 | Train Loss: 0.2110 | Val Loss: 0.2998 | Val Acc: 0.9216 | Time: 5.9s
Epoch 05/20 | Train Loss: 0.1673 | Val Loss: 0.2878 | Val Acc: 0.9169 | Time: 5.8s
Epoch 06/20 | Train Loss: 0.1371 | Val Loss: 0.3023 | Val Acc: 0.9290 | Time: 6.0s
Epoch 07/20 | Train Loss: 0.1126 | Val Loss: 0.3438 | Val Acc: 0.9324 | Time: 5.8s
Epoch 08/20 | Train Loss: 0.0963 | Val Loss: 0.3223 | Val Acc: 0.9338 | Time: 6.1s
Epoch 09/20 | Train Loss: 0.0818 | Val Loss: 0.3405 | Val Acc: 0.9335 | Time: 5.9s
Epoch 10/20 | Train Loss: 0.0688 | Val Loss: 0.3095 | Val Acc: 0.9344 | Time: 6.1s
Epoch 11/20 | Train Loss: 0.0617 | Val Loss: 0.3421 | Val Acc: 0.9365 | Time: 6.0s
Epoch 12/20 | Train Loss: 0.0528 | Val Loss: 0.3582 | Val Acc: 0.9365 | Time: 6.2s
Epoc

In [84]:
nlp = spacy.load('en_core_web_sm')

In [85]:
EPOCHS

20

In [86]:
def predict_ner(model, sentence, token2idx, idx2label, device):
  model.eval()
  tokens = [tok.text for tok in nlp.tokenizer(sentence)]
  token_indices = [token2idx.get(tok, UNK_IDX) for tok in tokens]
  src = torch.tensor(token_indices, dtype=torch.long).unsqueeze(0).to(device)
  mask = create_mask(src).to(device)
  with torch.no_grad():
    output = model(src, mask)
  predictions = output.argmax(dim=-1).squeeze(0)
  results = []
  for tok, pre_idx in zip(tokens, predictions):
    label = idx2label[pre_idx.item()]
    results.append((tok, label))
  return results


In [87]:
test_sentences = [
    "Barack Obama visited Google in New York.",
    "The European Union rejected the proposal.",
    "Apple CEO Tim Cook announced a new product in California.",
    "Manchester United signed a player from Brazil.",
]

In [88]:
for sent in test_sentences:
  results = predict_ner(model, sent, token2idx, idx2label, device)
  print(f"Sentence: {sent}")
  for word, label in results:
    if label != "0":
      print(f"   {word:20s} -> {label}")
  print()

Sentence: Barack Obama visited Google in New York.
   Barack               -> O
   Obama                -> O
   visited              -> O
   Google               -> O
   in                   -> O
   New                  -> B-ORG
   York                 -> I-ORG
   .                    -> O

Sentence: The European Union rejected the proposal.
   The                  -> O
   European             -> B-ORG
   Union                -> I-ORG
   rejected             -> O
   the                  -> O
   proposal             -> O
   .                    -> O

Sentence: Apple CEO Tim Cook announced a new product in California.
   Apple                -> I-ORG
   CEO                  -> O
   Tim                  -> B-PER
   Cook                 -> O
   announced            -> O
   a                    -> O
   new                  -> O
   product              -> O
   in                   -> O
   California           -> B-ORG
   .                    -> O

Sentence: Manchester United signed a player 

In [89]:
import json
import os

os.makedirs("ner_model", exist_ok=True)

# 1. Save model weights
torch.save(model.state_dict(), "ner_model/model.pt")

# 2. Save vocabulary
with open("ner_model/token2idx.json", "w") as f:
    json.dump(token2idx, f)

# 3. Save label mappings
with open("ner_model/label2idx.json", "w") as f:
    json.dump(label2idx, f)

with open("ner_model/idx2label.json", "w") as f:
    json.dump({str(k): v for k, v in idx2label.items()}, f)

# 4. Save hyperparameters
config = {
    "vocab_size": len(token2idx),
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "d_ff": D_FF,
    "n_layers": N_LAYERS,
    "num_labels": NUM_LABELS,
    "dropout": DROPOUT
}

with open("ner_model/config.json", "w") as f:
    json.dump(config, f)

print("Saved files:")
for f in os.listdir("ner_model"):
    size = os.path.getsize(f"ner_model/{f}") / (1024*1024)
    print(f"  {f} — {size:.2f} MB")

Saved files:
  token2idx.json — 0.39 MB
  config.json — 0.00 MB
  model.pt — 37.02 MB
  label2idx.json — 0.00 MB
  idx2label.json — 0.00 MB


In [90]:
for f in os.listdir("ner_model"):
    size = os.path.getsize(f"ner_model/{f}")
    print(f"  {f} — {size} bytes")

  token2idx.json — 410185 bytes
  config.json — 113 bytes
  model.pt — 38814483 bytes
  label2idx.json — 106 bytes
  idx2label.json — 124 bytes


In [91]:
app_code = '''import torch
import torch.nn as nn
import math
import json
import re
import gradio as gr

# ============================================================
# Model Architecture
# ============================================================

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, V)
        return output

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)
        Q = Q.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        output = self.W_o(attn_output)
        return output

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn_output = self.self_attention(x, x, x, mask)
        attn_output = self.dropout1(attn_output)
        x = self.norm1(x + attn_output)
        ff_output = self.feed_forward(x)
        ff_output = self.dropout2(ff_output)
        x = self.norm2(x + ff_output)
        return x

class NERTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, num_labels, dropout=0.1):
        super().__init__()
        self.token_embedding = TokenEmbedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, dropout=dropout)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.classifier = nn.Linear(d_model, num_labels)

    def forward(self, src, mask):
        x = self.token_embedding(src)
        x = self.positional_encoding(x)
        for layer in self.layers:
            x = layer(x, mask)
        output = self.classifier(x)
        return output

# ============================================================
# Masking
# ============================================================

PAD_IDX = 0
UNK_IDX = 1

def create_mask(src):
    mask = (src != PAD_IDX).unsqueeze(1).unsqueeze(2)
    return mask

# ============================================================
# Load Model
# ============================================================

device = torch.device('cpu')

with open("config.json", "r") as f:
    config = json.load(f)

with open("token2idx.json", "r") as f:
    token2idx = json.load(f)

with open("idx2label.json", "r") as f:
    idx2label = {int(k): v for k, v in json.load(f).items()}

model = NERTransformer(
    vocab_size=config["vocab_size"],
    d_model=config["d_model"],
    n_heads=config["n_heads"],
    d_ff=config["d_ff"],
    n_layers=config["n_layers"],
    num_labels=config["num_labels"],
    dropout=config["dropout"]
)

model.load_state_dict(torch.load("model.pt", map_location=device))
model.eval()

# ============================================================
# Tokenizer
# ============================================================

def tokenize(text):
    text = text.strip()
    tokens = re.findall(r"[\\w]+|[^\\s\\w]", text)
    return tokens

# ============================================================
# Color mapping for entities
# ============================================================

ENTITY_COLORS = {
    "PER": "#FF6B6B",
    "ORG": "#4ECDC4",
    "LOC": "#45B7D1",
    "MISC": "#96CEB4",
}

# ============================================================
# Prediction Function
# ============================================================

def predict_ner(sentence):
    if not sentence.strip():
        return "Please enter a sentence."

    tokens = tokenize(sentence)
    token_indices = [token2idx.get(tok, UNK_IDX) for tok in tokens]
    src = torch.tensor(token_indices, dtype=torch.long).unsqueeze(0)
    mask = create_mask(src)

    with torch.no_grad():
        output = model(src, mask)

    predictions = output.argmax(dim=-1).squeeze(0)

    # Build formatted output
    entities = []
    result_parts = []

    for tok, pred_idx in zip(tokens, predictions):
        label = idx2label[pred_idx.item()]
        if label != "O":
            entity_type = label.split("-")[1]
            entities.append(f"{tok} → {label}")
            result_parts.append((tok, entity_type))
        else:
            result_parts.append((tok, None))

    # Build highlighted text
    highlighted = []
    for tok, entity_type in result_parts:
        if entity_type:
            highlighted.append((tok, entity_type))
        else:
            highlighted.append((tok, None))

    # Build entity summary
    if entities:
        summary = "Detected entities:\\n" + "\\n".join(entities)
    else:
        summary = "No entities detected."

    return highlighted, summary

# ============================================================
# Gradio Interface
# ============================================================

examples = [
    "Barack Obama visited Google in New York.",
    "The European Union rejected the proposal.",
    "Apple CEO Tim Cook announced a new product in California.",
    "Manchester United signed a player from Brazil.",
    "The United Nations held a meeting in Geneva about climate change.",
]

demo = gr.Interface(
    fn=predict_ner,
    inputs=gr.Textbox(label="Enter a sentence", placeholder="Type an English sentence..."),
    outputs=[
        gr.HighlightedText(label="NER Tags", combine_adjacent=True,
                          color_map={"PER": "#FF6B6B", "ORG": "#4ECDC4", "LOC": "#45B7D1", "MISC": "#96CEB4"}),
        gr.Textbox(label="Detected Entities")
    ],
    title="Named Entity Recognition (NER)",
    description="Transformer encoder model built from scratch using PyTorch. Trained on CoNLL-2003 dataset (~14K sentences). Architecture: 3 encoder layers, 8 attention heads, d_model=256. Detects: PER (persons), ORG (organizations), LOC (locations), MISC (miscellaneous).",
    examples=examples,
    theme=gr.themes.Soft()
)

if __name__ == "__main__":
    demo.launch()
'''

with open("ner_model/app.py", "w") as f:
    f.write(app_code)

# Requirements
with open("ner_model/requirements.txt", "w") as f:
    f.write("torch\naudioop-lts\n")

# README
readme = """---
title: Named Entity Recognition
emoji: 🏷️
colorFrom: green
colorTo: blue
sdk: gradio
sdk_version: "5.12.0"
app_file: app.py
pinned: false
---

# Named Entity Recognition (NER)

A Transformer encoder model built from scratch using PyTorch, trained on the CoNLL-2003 dataset.

## Architecture
- **Type:** Transformer Encoder + Classification Head
- **Encoder Layers:** 3
- **Attention Heads:** 8
- **Model Dimension:** 256
- **Feed-Forward Dimension:** 1024
- **Parameters:** ~8.4M
- **Dataset:** CoNLL-2003 (~14K sentences)

## Entity Types
- **PER** — Person names
- **ORG** — Organizations
- **LOC** — Locations
- **MISC** — Miscellaneous entities

## How It Works
Every component — embeddings, positional encoding, multi-head attention, feed-forward layers — was implemented from scratch. No pre-trained weights or HuggingFace model classes were used. Only the encoder part of the Transformer is used since NER is a token classification task.
"""

with open("ner_model/README.md", "w") as f:
    f.write(readme)

print("All files created!")
for f in os.listdir("ner_model"):
    print(f"  {f}")

All files created!
  token2idx.json
  requirements.txt
  config.json
  model.pt
  app.py
  label2idx.json
  README.md
  idx2label.json


In [92]:
%cd /content

/content


In [93]:
!git clone https://huggingface.co/spaces/Abhisek987/ner-transformer hf_ner_repo
%cd hf_ner_repo

Cloning into 'hf_ner_repo'...
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 16 (delta 0), reused 0 (delta 0), pack-reused 4 (from 1)
Receiving objects: 100% (16/16), 172.86 KiB | 5.76 MiB/s, done.
/content/hf_ner_repo


In [94]:
!ls -la

total 38360
drwxr-xr-x 3 root root     4096 Apr 14 11:12 .
drwxr-xr-x 1 root root     4096 Apr 14 11:12 ..
-rw-r--r-- 1 root root     8632 Apr 14 11:12 app.py
-rw-r--r-- 1 root root      113 Apr 14 11:12 config.json
drwxr-xr-x 9 root root     4096 Apr 14 11:12 .git
-rw-r--r-- 1 root root     1519 Apr 14 11:12 .gitattributes
-rw-r--r-- 1 root root      124 Apr 14 11:12 idx2label.json
-rw-r--r-- 1 root root      106 Apr 14 11:12 label2idx.json
-rw-r--r-- 1 root root 38814483 Apr 14 11:12 model.pt
-rw-r--r-- 1 root root      966 Apr 14 11:12 README.md
-rw-r--r-- 1 root root       18 Apr 14 11:12 requirements.txt
-rw-r--r-- 1 root root   410185 Apr 14 11:12 token2idx.json


In [95]:
# Create train.py
train_code = '''import torch
import torch.nn as nn
import torch.optim as optim
import math
import time
import os
import json
import urllib.request
from collections import Counter
from torch.utils.data import Dataset, DataLoader

# ============================================================
# Device Setup
# ============================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ============================================================
# Download and Load CoNLL-2003 Dataset
# ============================================================

os.makedirs("data", exist_ok=True)

base_url = "https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/"

files = {
    "train.txt": "eng.train",
    "valid.txt": "eng.testa",
    "test.txt": "eng.testb"
}

for local_name, remote_name in files.items():
    path = f"data/{local_name}"
    if not os.path.exists(path):
        print(f"Downloading {remote_name}...")
        urllib.request.urlretrieve(base_url + remote_name, path)

print("Dataset ready!")

def load_conll(filepath):
    sentences = []
    labels = []
    current_tokens = []
    current_labels = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line == "" or line.startswith("-DOCSTART-"):
                if current_tokens:
                    sentences.append(current_tokens)
                    labels.append(current_labels)
                    current_tokens = []
                    current_labels = []
                continue
            parts = line.split()
            current_tokens.append(parts[0])
            current_labels.append(parts[3])
        if current_tokens:
            sentences.append(current_tokens)
            labels.append(current_labels)
    return sentences, labels

train_sentences, train_labels = load_conll("data/train.txt")
val_sentences, val_labels = load_conll("data/valid.txt")
test_sentences, test_labels = load_conll("data/test.txt")

print(f"Train: {len(train_sentences)} sentences")
print(f"Val:   {len(val_sentences)} sentences")
print(f"Test:  {len(test_sentences)} sentences")

# ============================================================
# Convert IOB1 to IOB2
# ============================================================

def convert_to_iob2(labels):
    new_labels = []
    for i, label in enumerate(labels):
        if label.startswith("I-"):
            if i == 0 or labels[i-1] == "O" or labels[i-1].split("-")[1] != label.split("-")[1]:
                new_labels.append("B-" + label.split("-")[1])
            else:
                new_labels.append(label)
        else:
            new_labels.append(label)
    return new_labels

train_labels = [convert_to_iob2(labels) for labels in train_labels]
val_labels = [convert_to_iob2(labels) for labels in val_labels]
test_labels = [convert_to_iob2(labels) for labels in test_labels]

# ============================================================
# Label Mappings
# ============================================================

label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
label2idx = {label: idx for idx, label in enumerate(label_list)}
idx2label = {idx: label for label, idx in label2idx.items()}
NUM_LABELS = len(label_list)

# ============================================================
# Vocabulary
# ============================================================

PAD_IDX = 0
UNK_IDX = 1

def build_vocab(sentences, min_freq=1):
    counter = Counter()
    for sent in sentences:
        counter.update(sent)
    token2idx = {'<pad>': 0, '<unk>': 1}
    for word, freq in counter.items():
        if freq >= min_freq and word not in token2idx:
            token2idx[word] = len(token2idx)
    idx2token = {idx: tok for tok, idx in token2idx.items()}
    return token2idx, idx2token

token2idx, idx2token = build_vocab(train_sentences, min_freq=1)
print(f"Vocab size: {len(token2idx)}")

# ============================================================
# Dataset and DataLoader
# ============================================================

LABEL_PAD_IDX = -100

class NERDataset(Dataset):
    def __init__(self, sentences, labels, token2idx, label2idx):
        self.sentences = sentences
        self.labels = labels
        self.token2idx = token2idx
        self.label2idx = label2idx

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        label = self.labels[idx]
        token_indices = [self.token2idx.get(word, UNK_IDX) for word in sentence]
        label_indices = [self.label2idx[lab] for lab in label]
        return torch.tensor(token_indices, dtype=torch.long), torch.tensor(label_indices, dtype=torch.long)

def collate_fn(batch):
    tokens_batch, labels_batch = zip(*batch)
    max_len = max(len(seq) for seq in tokens_batch)
    tokens_padded = []
    labels_padded = []
    for tokens, labels in zip(tokens_batch, labels_batch):
        token_padding = torch.full((max_len - len(tokens),), PAD_IDX, dtype=torch.long)
        label_padding = torch.full((max_len - len(labels),), LABEL_PAD_IDX, dtype=torch.long)
        tokens_padded.append(torch.cat([tokens, token_padding]))
        labels_padded.append(torch.cat([labels, label_padding]))
    return torch.stack(tokens_padded), torch.stack(labels_padded)

BATCH_SIZE = 64

train_dataset = NERDataset(train_sentences, train_labels, token2idx, label2idx)
val_dataset = NERDataset(val_sentences, val_labels, token2idx, label2idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# ============================================================
# Model Components
# ============================================================

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, V)
        return output

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)
        Q = Q.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        output = self.W_o(attn_output)
        return output

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn_output = self.self_attention(x, x, x, mask)
        attn_output = self.dropout1(attn_output)
        x = self.norm1(x + attn_output)
        ff_output = self.feed_forward(x)
        ff_output = self.dropout2(ff_output)
        x = self.norm2(x + ff_output)
        return x

class NERTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, num_labels, dropout=0.1):
        super().__init__()
        self.token_embedding = TokenEmbedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, dropout=dropout)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.classifier = nn.Linear(d_model, num_labels)

    def forward(self, src, mask):
        x = self.token_embedding(src)
        x = self.positional_encoding(x)
        for layer in self.layers:
            x = layer(x, mask)
        output = self.classifier(x)
        return output

# ============================================================
# Masking
# ============================================================

def create_mask(src):
    mask = (src != PAD_IDX).unsqueeze(1).unsqueeze(2)
    return mask

# ============================================================
# Training and Evaluation
# ============================================================

def train_one_epoch(model, dataloader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    total_tokens = 0
    for tokens, labels in dataloader:
        tokens = tokens.to(device)
        labels = labels.to(device)
        mask = create_mask(tokens).to(device)
        output = model(tokens, mask)
        output = output.reshape(-1, output.size(-1))
        labels = labels.reshape(-1)
        loss = loss_fn(output, labels)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        non_pad = (labels != LABEL_PAD_IDX).sum().item()
        total_loss += loss.item() * non_pad
        total_tokens += non_pad
    return total_loss / total_tokens

def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0
    total_tokens = 0
    with torch.no_grad():
        for tokens, labels in dataloader:
            tokens = tokens.to(device)
            labels = labels.to(device)
            mask = create_mask(tokens).to(device)
            output = model(tokens, mask)
            output = output.reshape(-1, output.size(-1))
            labels = labels.reshape(-1)
            loss = loss_fn(output, labels)
            non_pad = (labels != LABEL_PAD_IDX).sum().item()
            total_loss += loss.item() * non_pad
            total_tokens += non_pad
    return total_loss / total_tokens

def compute_accuracy(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for tokens, labels in dataloader:
            tokens = tokens.to(device)
            labels = labels.to(device)
            mask = create_mask(tokens).to(device)
            output = model(tokens, mask)
            predictions = output.argmax(dim=-1)
            non_pad_mask = (labels != LABEL_PAD_IDX)
            correct += (predictions[non_pad_mask] == labels[non_pad_mask]).sum().item()
            total += non_pad_mask.sum().item()
    return correct / total

# ============================================================
# Hyperparameters
# ============================================================

D_MODEL = 256
N_HEADS = 8
D_FF = 1024
N_LAYERS = 3
DROPOUT = 0.1
LEARNING_RATE = 3e-4
EPOCHS = 20

# ============================================================
# Create Model
# ============================================================

model = NERTransformer(
    vocab_size=len(token2idx),
    d_model=D_MODEL,
    n_heads=N_HEADS,
    d_ff=D_FF,
    n_layers=N_LAYERS,
    num_labels=NUM_LABELS,
    dropout=DROPOUT
).to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=LABEL_PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.98), eps=1e-9)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

# ============================================================
# Training Loop
# ============================================================

print("\\nStarting training...\\n")

for epoch in range(EPOCHS):
    start_time = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    val_loss = evaluate(model, val_loader, loss_fn, device)
    val_acc = compute_accuracy(model, val_loader, device)
    elapsed = time.time() - start_time
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.4f} | "
          f"Time: {elapsed:.1f}s")

# ============================================================
# Save Model
# ============================================================

save_dir = "ner_model"
os.makedirs(save_dir, exist_ok=True)

torch.save(model.state_dict(), f"{save_dir}/model.pt")

with open(f"{save_dir}/token2idx.json", "w") as f:
    json.dump(token2idx, f)

with open(f"{save_dir}/label2idx.json", "w") as f:
    json.dump(label2idx, f)

with open(f"{save_dir}/idx2label.json", "w") as f:
    json.dump({str(k): v for k, v in idx2label.items()}, f)

config = {
    "vocab_size": len(token2idx),
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "d_ff": D_FF,
    "n_layers": N_LAYERS,
    "num_labels": NUM_LABELS,
    "dropout": DROPOUT
}

with open(f"{save_dir}/config.json", "w") as f:
    json.dump(config, f)

print(f"\\nModel saved to {save_dir}/")

# ============================================================
# Test Predictions
# ============================================================

import re

def tokenize(text):
    text = text.strip()
    tokens = re.findall(r"[\\w]+|[^\\s\\w]", text)
    return tokens

def predict_ner(sentence):
    model.eval()
    tokens = tokenize(sentence)
    token_indices = [token2idx.get(tok, UNK_IDX) for tok in tokens]
    src = torch.tensor(token_indices, dtype=torch.long).unsqueeze(0).to(device)
    mask = create_mask(src).to(device)
    with torch.no_grad():
        output = model(src, mask)
    predictions = output.argmax(dim=-1).squeeze(0)
    results = []
    for tok, pred_idx in zip(tokens, predictions):
        label = idx2label[pred_idx.item()]
        results.append((tok, label))
    return results

print("\\n=== Sample Predictions ===\\n")

test_sentences = [
    "Barack Obama visited Google in New York.",
    "The European Union rejected the proposal.",
    "Apple CEO Tim Cook announced a new product in California.",
    "Manchester United signed a player from Brazil.",
]

for sent in test_sentences:
    results = predict_ner(sent)
    print(f"Sentence: {sent}")
    for word, label in results:
        if label != "O":
            print(f"  {word:20s} -> {label}")
    print()
'''

with open("train.py", "w") as f:
    f.write(train_code)

print("train.py created!")

train.py created!
